In [1]:
import numpy as np
import pickle
import os
import time

def moss_multiclass_mn(
    n_samples: int,
    alpha: np.ndarray,
    merge: float,
    eps: float = 1e-3
):
    n_classes = len(alpha)
    merge = np.clip(merge, 0.0, 1.0)

    # Controle de variância (análoga ao scale do Dirichlet)
    base_var = 0.01
    max_var = 0.5
    var = base_var * (1 - merge) + max_var * merge

    scores = np.zeros((n_samples, n_classes))

    n_per_class = np.floor(n_samples * alpha).astype(int)
    n_per_class[-1] = n_samples - n_per_class[:-1].sum()

    idx = 0
    for c in range(n_classes):
        # Centrôide no vértice do simplex
        mean = np.zeros(n_classes)
        mean[c] = 1.0

        cov = np.eye(n_classes) * var

        samples = np.random.multivariate_normal(
            mean=mean,
            cov=cov,
            size=n_per_class[c]
        )

        # Projeção no simplex
        samples = np.abs(samples) + eps
        samples /= samples.sum(axis=1, keepdims=True)

        scores[idx:idx + n_per_class[c]] = samples
        idx += n_per_class[c]

    np.random.shuffle(scores)
    return scores

def gerar_distribuicoes_moss_multiclasse(
    n_samples,
    n_classes,
    n_prevalences,
    n_merges,
    n_curves,
    save_path
):
    # Prevalências continuam Dirichlet (correto!)
    prevalences = np.random.dirichlet(
        alpha=np.ones(n_classes),
        size=n_prevalences
    )

    merges = np.linspace(0.0, 1.0, n_merges)
    synthetic_distributions = {}
    total = len(prevalences) * len(merges)
    count = 0

    print(f"\n🚀 Gerando MoSS-MN para {n_classes} CLASSES -> {save_path}")
    start_global = time.perf_counter()

    for alpha in prevalences:
        alpha_key = tuple(np.round(alpha, 4))

        for merge in merges:
            curves = [
                moss_multiclass_mn(
                    n_samples,
                    alpha,
                    merge
                )
                for _ in range(n_curves)
            ]

            synthetic_distributions[(alpha_key, round(merge, 4))] = curves
            count += 1

            if count % 50 == 0 or count == total:
                print(f"   Progresso: [{count}/{total}] blocos concluídos...")

    with open(save_path, "wb") as f:
        pickle.dump(
            synthetic_distributions,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    total_runtime = time.perf_counter() - start_global
    print(f"✔ Finalizado {n_classes} classes em {total_runtime/60:.2f} min\n")

if __name__ == "__main__":
    output_dir = "moss_outputs_mn"
    os.makedirs(output_dir, exist_ok=True)

    base_config = dict(
        n_samples=100,
        n_prevalences=15,
        n_merges=15,
        n_curves=20
    )

    classes_para_gerar = [3, 4, 5, 6, 7, 8, 10, 11, 15, 26]

    for c in classes_para_gerar:
        file_path = os.path.join(
            output_dir,
            f"moss_m_mn_{c}.pkl"
        )

        gerar_distribuicoes_moss_multiclasse(
            n_classes=c,
            save_path=file_path,
            **base_config
        )

    print("🏁 Todos os arquivos MoSS-MN multiclasse foram gerados com sucesso.")


🚀 Gerando MoSS-MN para 3 CLASSES -> moss_outputs_mn/moss_m_mn_3.pkl
   Progresso: [50/225] blocos concluídos...
   Progresso: [100/225] blocos concluídos...
   Progresso: [150/225] blocos concluídos...
   Progresso: [200/225] blocos concluídos...
   Progresso: [225/225] blocos concluídos...
✔ Finalizado 3 classes em 0.01 min


🚀 Gerando MoSS-MN para 4 CLASSES -> moss_outputs_mn/moss_m_mn_4.pkl
   Progresso: [50/225] blocos concluídos...
   Progresso: [100/225] blocos concluídos...
   Progresso: [150/225] blocos concluídos...
   Progresso: [200/225] blocos concluídos...
   Progresso: [225/225] blocos concluídos...
✔ Finalizado 4 classes em 0.01 min


🚀 Gerando MoSS-MN para 5 CLASSES -> moss_outputs_mn/moss_m_mn_5.pkl
   Progresso: [50/225] blocos concluídos...
   Progresso: [100/225] blocos concluídos...
   Progresso: [150/225] blocos concluídos...
   Progresso: [200/225] blocos concluídos...
   Progresso: [225/225] blocos concluídos...
✔ Finalizado 5 classes em 0.01 min


🚀 Gerando Mo

In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Carregar os dados
df = pd.read_csv("m_30_bcts.csv")

# 2. Configurações de layout
datasets = df['dataset'].unique()
n_datasets = len(datasets)
# Criamos uma subfigura por dataset, em uma única coluna
fig = make_subplots(
    rows=n_datasets, cols=1, 
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.02 # Espaço curto entre os gráficos
)

# 3. Iterar e adicionar cada gráfico com sua própria ordem
for i, ds in enumerate(datasets, 1):
    df_ds = df[df['dataset'] == ds].copy()
    
    # Calcular a ordem local (pela mediana do erro neste dataset)
    ordem_local = df_ds.groupby("modelo")["erro"].median().sort_values().index.tolist()
    
    # Adicionar um boxplot para cada modelo, seguindo a ordem local
    for modelo in ordem_local:
        df_mod = df_ds[df_ds['modelo'] == modelo]
        fig.add_trace(
            go.Box(
                y=df_mod['erro'],
                name=modelo,
                boxpoints='outliers',
                legendgroup=modelo,
                showlegend=(i == 1) # Só mostra a legenda no primeiro gráfico
            ),
            row=i, col=1
        )

# 4. Ajustes finais de tamanho e estética
fig.update_layout(
    height=n_datasets * 400, # 300px para cada dataset
    template="plotly_white",
    title_text="Performance Local: Modelos Ordenados do Melhor para o Pior por Dataset",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

# Deixar os eixos X independentes para cada subgráfico respeitar sua ordem
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="MAE")
#fig.write_html("meu_resultado_ordenado.html")
fig.show()

In [5]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# 1. Carregar os dados
# =========================
df_mn = pd.read_csv("mn_30_bcts_perclass.csv")
df_d  = pd.read_csv("../exp_012/d_30_bcts_perclass.csv")

# =========================
# 2. Filtrar apenas MoSS
# =========================
df_mn["modelo"] = df_mn["modelo"].astype(str)
df_d["modelo"]  = df_d["modelo"].astype(str)

df_mn = df_mn[df_mn["modelo"].str.startswith("MoSS_")].copy()
df_d  = df_d[df_d["modelo"].str.startswith("MoSS_")].copy()

df_mn["origem"] = "MN"
df_d["origem"]  = "D"

# =========================
# 3. Unir os dados
# =========================
df = pd.concat([df_mn, df_d], ignore_index=True)

datasets = df["dataset"].unique()
n_datasets = len(datasets)

if n_datasets == 0:
    raise ValueError("Nenhum dataset encontrado após o filtro de MoSS.")

# =========================
# 4. Criar subplots
# =========================
fig = make_subplots(
    rows=n_datasets,
    cols=1,
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.03
)

# =========================
# 5. Plotar
# =========================
for i, ds in enumerate(datasets, 1):
    df_ds = df[df["dataset"] == ds].copy()

    # Ordem dos MODELOS (melhor → pior)
    ordem_modelos = (
        df_ds.groupby("modelo")["erro"]
        .median()
        .sort_values()
        .index
        .tolist()
    )

    x_order = []

    for modelo in ordem_modelos:
        df_mod = df_ds[df_ds["modelo"] == modelo]

        med_mn = df_mod[df_mod["origem"] == "MN"]["erro"].median()
        med_d  = df_mod[df_mod["origem"] == "D"]["erro"].median()

        # 🔥 decidir quem fica à esquerda
        if med_mn <= med_d:
            origens_ordenadas = ["MN", "D"]
        else:
            origens_ordenadas = ["D", "MN"]

        # registrar ordem no eixo X
        for origem in origens_ordenadas:
            x_order.append(f"{modelo} ({origem})")

            df_plot = df_mod[df_mod["origem"] == origem]
            if df_plot.empty:
                continue

            fig.add_trace(
                go.Box(
                    y=df_plot["erro"],
                    name=f"{modelo} ({origem})",
                    legendgroup=f"{modelo}_{origem}",
                    boxpoints="outliers",
                    showlegend=(i == 1)
                ),
                row=i,
                col=1
            )

    # 🔒 Forçar ordem correta no eixo X
    fig.update_xaxes(
        categoryorder="array",
        categoryarray=x_order,
        row=i,
        col=1
    )


# =========================
# 6. Layout final
# =========================
fig.update_layout(
    height=n_datasets * 450,
    template="plotly_white",
    title_text="Comparação MN × D — MoSS (30 BCTs por classe)",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

fig.update_yaxes(title_text="MAE")
fig.update_xaxes(showgrid=False)

fig.show()

In [4]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# 1. Carregar os dados
# =========================
df_mn = pd.read_csv("mn_30_bcts.csv")
df_d  = pd.read_csv("../exp_011/d_30_bcts.csv")

# =========================
# 2. Filtrar apenas MoSS
# =========================
df_mn["modelo"] = df_mn["modelo"].astype(str)
df_d["modelo"]  = df_d["modelo"].astype(str)

df_mn = df_mn[df_mn["modelo"].str.startswith("MoSS_")].copy()
df_d  = df_d[df_d["modelo"].str.startswith("MoSS_")].copy()

df_mn["origem"] = "MN"
df_d["origem"]  = "D"

# =========================
# 3. Unir os dados
# =========================
df = pd.concat([df_mn, df_d], ignore_index=True)

datasets = df["dataset"].unique()
n_datasets = len(datasets)

if n_datasets == 0:
    raise ValueError("Nenhum dataset encontrado após o filtro de MoSS.")

# =========================
# 4. Criar subplots
# =========================
fig = make_subplots(
    rows=n_datasets,
    cols=1,
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.03
)

# =========================
# 5. Plotar
# =========================
for i, ds in enumerate(datasets, 1):
    df_ds = df[df["dataset"] == ds].copy()

    # Ordem dos MODELOS (melhor → pior)
    ordem_modelos = (
        df_ds.groupby("modelo")["erro"]
        .median()
        .sort_values()
        .index
        .tolist()
    )

    x_order = []

    for modelo in ordem_modelos:
        df_mod = df_ds[df_ds["modelo"] == modelo]

        med_mn = df_mod[df_mod["origem"] == "MN"]["erro"].median()
        med_d  = df_mod[df_mod["origem"] == "D"]["erro"].median()

        # 🔥 decidir quem fica à esquerda
        if med_mn <= med_d:
            origens_ordenadas = ["MN", "D"]
        else:
            origens_ordenadas = ["D", "MN"]

        # registrar ordem no eixo X
        for origem in origens_ordenadas:
            x_order.append(f"{modelo} ({origem})")

            df_plot = df_mod[df_mod["origem"] == origem]
            if df_plot.empty:
                continue

            fig.add_trace(
                go.Box(
                    y=df_plot["erro"],
                    name=f"{modelo} ({origem})",
                    legendgroup=f"{modelo}_{origem}",
                    boxpoints="outliers",
                    showlegend=(i == 1)
                ),
                row=i,
                col=1
            )

    # 🔒 Forçar ordem correta no eixo X
    fig.update_xaxes(
        categoryorder="array",
        categoryarray=x_order,
        row=i,
        col=1
    )


# =========================
# 6. Layout final
# =========================
fig.update_layout(
    height=n_datasets * 450,
    template="plotly_white",
    title_text="Comparação MN × D — MoSS (30 BCTs por classe)",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

fig.update_yaxes(title_text="MAE")
fig.update_xaxes(showgrid=False)

fig.show()

In [5]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Carregar os dados
df = pd.read_csv("mn_30_bcts.csv")

# 2. Configurações de layout
datasets = df['dataset'].unique()
n_datasets = len(datasets)
# Criamos uma subfigura por dataset, em uma única coluna
fig = make_subplots(
    rows=n_datasets, cols=1, 
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.02 # Espaço curto entre os gráficos
)

# 3. Iterar e adicionar cada gráfico com sua própria ordem
for i, ds in enumerate(datasets, 1):
    df_ds = df[df['dataset'] == ds].copy()
    
    # Calcular a ordem local (pela mediana do erro neste dataset)
    ordem_local = df_ds.groupby("modelo")["erro"].median().sort_values().index.tolist()
    
    # Adicionar um boxplot para cada modelo, seguindo a ordem local
    for modelo in ordem_local:
        df_mod = df_ds[df_ds['modelo'] == modelo]
        fig.add_trace(
            go.Box(
                y=df_mod['erro'],
                name=modelo,
                boxpoints='outliers',
                legendgroup=modelo,
                showlegend=(i == 1) # Só mostra a legenda no primeiro gráfico
            ),
            row=i, col=1
        )

# 4. Ajustes finais de tamanho e estética
fig.update_layout(
    height=n_datasets * 400, # 300px para cada dataset
    template="plotly_white",
    title_text="Performance Local: Modelos Ordenados do Melhor para o Pior por Dataset",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

# Deixar os eixos X independentes para cada subgráfico respeitar sua ordem
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="MAE")
#fig.write_html("meu_resultado_ordenado.html")
fig.show()